# 0.8+ Audio Deepfake Detection Pipeline

GPU T4 연결 후 → **런타임 → 모두 실행** 하면 끝.

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install librosa soundfile transformers accelerate demucs panns-inference onnxruntime-gpu datasets huggingface_hub scikit-learn scipy edge-tts
!apt -qq install -y ffmpeg > /dev/null
print('Done')

In [ ]:
!git clone https://github.com/jogwangjo/da1.git /content/da1
%cd /content/da1
!pwd && ls

In [ ]:
import os
required = ['scripts/build_train_data.py', 'scripts/train_raptor.py', 'submit/script_v2.py', 'data/sample_submission.csv']
for f in required:
    print(f'  {"OK" if os.path.exists(f) else "MISSING"}: {f}')

In [ ]:
import os
from huggingface_hub import snapshot_download

model_dir = 'submit/model'
os.makedirs(model_dir, exist_ok=True)

# DF-Arena 1B
df_dir = f'{model_dir}/df_arena_1b'
if not os.path.exists(f'{df_dir}/pytorch_model.bin'):
    print('Downloading DF-Arena 1B...')
    snapshot_download('shreshthgupta/DF-Arena-1B-Antispoofing', local_dir=df_dir)

# HTDemucs
htd_dir = f'{model_dir}/htdemucs'
if not os.path.exists(f'{htd_dir}/955717e8-8726e21a.th'):
    print('Downloading HTDemucs...')
    from demucs.pretrained import get_model
    import torch
    m = get_model('htdemucs')
    os.makedirs(htd_dir, exist_ok=True)
    torch.save(m.state_dict(), f'{htd_dir}/955717e8-8726e21a.th')

# SONICS
for name in ['sonics-alpha-5s', 'sonics-beta-5s']:
    if not os.path.exists(f'{model_dir}/{name}/pytorch_model.bin'):
        print(f'Downloading {name}...')
        snapshot_download('sagierte/sonics', allow_patterns=[f'{name}/*'], local_dir=model_dir)

print('Models done!')

In [ ]:
# LibriSpeech 다운로드
!python scripts/download_librispeech.py

# 학습 데이터 구축
!python scripts/build_train_data.py --out train_data --auto-download --max-voice-real 2000 --max-voice-fake 3000 --max-music-real 500 --max-music-fake 2000

In [ ]:
!python scripts/train_raptor.py --train train_data/manifest_train.csv --val train_data/manifest_val.csv --backbone utter-project/mHuBERT-147 --out runs/raptor_v1 --epochs 20 --bs 24 --lr 1e-6 --lr-head 3e-4 --consistency-w 0.25 --p-aug 0.5

In [ ]:
import pandas as pd, os
log_path = 'runs/raptor_v1/log.csv'
if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    print(log.to_string(index=False))
    print(f'\nBest val EER: {log["val_eer"].min():.4f}')
else:
    print(f'ERROR: {log_path} not found!')

In [ ]:
!cp runs/raptor_v1/best.pth submit/model/raptor_best.pth
!python submit/script_v2.py --test-dir data/test --sample-submission data/sample_submission.csv --output output/submission.csv --device cuda --tta 2

In [ ]:
!python scripts/build_submit_zip.py --script submit/script_v2.py --output submit.zip

In [ ]:
from google.colab import files
files.download('submit.zip')
files.download('output/submission.csv')